<a href="https://colab.research.google.com/github/ananthurajeev/GEO5017/blob/main/GEO5017_Bonus_Task_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEO5017 — BONUS: Localization + Classification



### Prerequisite
Top-100 images must already exist in `OUTPUT_DIR/top100_waste_detections/`

## STEP 1 — Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── EDIT THIS to match your setup ────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/GEO5017/outputs'
# ─────────────────────────────────────────────────────────────────────────────

TOP100_DIR       = os.path.join(OUTPUT_DIR, 'top100_waste_detections')
BONUS_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'bonus_localization')
os.makedirs(BONUS_OUTPUT_DIR, exist_ok=True)

images = [f for f in os.listdir(TOP100_DIR) if f.lower().endswith('.jpg')]
print(f'Top-100 folder   : {TOP100_DIR}')
print(f'Bonus output dir : {BONUS_OUTPUT_DIR}')
print(f'Images found     : {len(images)}')

## STEP 2 — Install YOLO World


In [ ]:
!pip install -q ultralytics

## STEP 3 — Load YOLO World model

In [ ]:
import torch
from ultralytics import YOLOWorld

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# yolov8s-worldv2.pt is the small, fast version — good for Colab T4
model = YOLOWorld('yolov8s-worldv2.pt')
print('YOLO World loaded.')

## STEP 4 — Define waste categories



In [ ]:
# Waste categories — these are passed as text classes to YOLO World
WASTE_CLASSES = [
    'litter',
    'garbage bag',
    'bulky waste',
    'waste bin',
]

CONF_THRESHOLD = 0.10   # lower = more detections; raise to 0.20 to reduce false positives

# Set the classes on the model
model.set_classes(WASTE_CLASSES)

# Color per class for drawing (BGR for OpenCV)
CLASS_COLORS = {
    'litter'      : (80,  80,  255),   # red
    'garbage bag' : (80,  200, 80),    # green
    'bulky waste' : (255, 80,  80),    # blue
    'waste bin'   : (200, 80,  200),   # purple
}

print('Classes set:')
for i, cls in enumerate(WASTE_CLASSES):
    print(f'  [{i}] {cls}')

## STEP 5 — Run detection on all Top-100 images

In [ ]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

all_detections = []
CONF_THRESHOLD = 0.05  # very low — catch more detections

# Run each class separately for better recall
CLASS_LIST = [
    ('litter and rubbish on ground',    'litter',       (80,  80,  255)),
    ('trash bag and garbage bag',        'garbage_bag',  (80,  200, 80)),
    ('bulky waste number of waste together in one place',       'bulky_waste',  (255, 80,  80)),
    ('garbage bin waste container',      'waste_bin',    (200, 80,  200)),
]

for img_name in tqdm(images, desc='Detecting waste'):
    img_path = os.path.join(TOP100_DIR, img_name)
    frame = cv2.imread(img_path)
    h, w = frame.shape[:2]

    for prompt, category, color in CLASS_LIST:
        # Set one class at a time
        model.set_classes([prompt])
        results = model.predict(img_path, conf=CONF_THRESHOLD, verbose=False)[0]

        if results.boxes is not None and len(results.boxes) > 0:
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                conf = float(box.conf[0])

                # Draw box
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                label = f'{category} {conf:.2f}'
                label_w = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0][0]
                cv2.rectangle(frame, (x1, y1-20), (x1+label_w+4, y1), color, -1)
                cv2.putText(frame, label, (x1+2, y1-5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

                all_detections.append({
                    'filename': img_name,
                    'category': category,
                    'confidence': round(conf, 4),
                    'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                })

    # Save annotated image
    cv2.imwrite(os.path.join(BONUS_OUTPUT_DIR, img_name), frame)

print(f'Total boxes: {len(all_detections)}')
print(f'Images with detections: {len(set(d["filename"] for d in all_detections))}')

## STEP 6 — Save detections to CSV

In [ ]:
det_df = pd.DataFrame(all_detections)

if len(det_df) > 0:
    csv_path = os.path.join(BONUS_OUTPUT_DIR, 'detections.csv')
    det_df.to_csv(csv_path, index=False)

    print('Detections by category:')
    print(det_df['category'].value_counts().to_string())
    print(f'\nAverage confidence : {det_df["confidence"].mean():.3f}')
    print(f'\n✅ CSV saved to: {csv_path}')
else:
    print('⚠️  No detections found.')
    print('Try lowering CONF_THRESHOLD to 0.05 in STEP 4 and re-running STEP 5.')

## STEP 7 — Visualize Top-10 annotated images

In [ ]:
import matplotlib.pyplot as plt

# Show images with most detections first
if len(det_df) > 0:
    show_images = det_df.groupby('filename').size().sort_values(ascending=False).head(10).index.tolist()
else:
    show_images = images[:10]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, fname in zip(axes.flatten(), show_images):
    img = cv2.imread(os.path.join(BONUS_OUTPUT_DIR, fname))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    n = len(det_df[det_df['filename'] == fname]) if len(det_df) > 0 else 0
    ax.imshow(img)
    ax.set_title(f'{n} box(es)', fontsize=8)
    ax.axis('off')

plt.suptitle('Top-10 annotated images (most detections first)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(BONUS_OUTPUT_DIR, 'sample_detections.png'), dpi=150)
plt.show()